# 03: Docker & FastAPI for ML Model Serving

**Module 3: MLOps & Production Systems | Week 12**

---

## Learning Objectives

1. Understand Docker fundamentals and why containerization is essential for ML reproducibility
2. Write production-quality Dockerfiles for ML applications with multi-stage builds
3. Build REST APIs with FastAPI including type-safe request/response models
4. Validate ML model inputs using Pydantic with custom constraints and validators
5. Serve trained ML models via prediction endpoints (single and batch)
6. Implement middleware for logging, timing, error handling, and CORS
7. Add health checks and metrics endpoints for production monitoring
8. Orchestrate multi-service deployments with Docker Compose
9. Write comprehensive API test suites using FastAPI's TestClient
10. Apply best practices for containerized ML model serving

## Resources

- [Docker Documentation](https://docs.docker.com/)
- [FastAPI Documentation](https://fastapi.tiangolo.com/)
- [Pydantic V2 Documentation](https://docs.pydantic.dev/latest/)
- [Docker for Data Science (O'Reilly)](https://www.oreilly.com/library/view/docker-for-data/9781484296165/)
- [Deploying ML Models with FastAPI](https://fastapi.tiangolo.com/tutorial/)
- [12-Factor App Methodology](https://12factor.net/)

In [ ]:
# Standard imports
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
import json
import time
import os
import subprocess
import logging
from datetime import datetime
from typing import List, Optional, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed

# ML imports
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# FastAPI imports (may need: pip install fastapi httpx)
try:
    from fastapi import FastAPI, HTTPException, Request
    from fastapi.testclient import TestClient
    from fastapi.middleware.cors import CORSMiddleware
    from pydantic import BaseModel, Field, field_validator
    print("FastAPI and Pydantic imported successfully.")
except ImportError as e:
    print(f"Import error: {e}")
    print("Install with: pip install 'fastapi[all]' httpx")

try:
    import httpx
    print("httpx imported successfully.")
except ImportError:
    print("httpx not installed. Install with: pip install httpx")

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print(f"\nNumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Python: {sys.version}")

---

## Section 1: Why Docker for ML

### The "Works on My Machine" Problem

Machine learning projects are notoriously difficult to reproduce. The challenges include:

| Problem | Example | Impact |
|---------|---------|--------|
| Python version mismatch | Model trained on 3.9, deployed on 3.11 | Pickle deserialization failures |
| CUDA / GPU driver conflicts | Training with CUDA 11.8, server has 12.1 | Silent numerical differences |
| Library version drift | scikit-learn 1.2 vs 1.3 API changes | Broken feature pipelines |
| OS-level dependencies | libgomp, libblas missing on production | Runtime crashes |
| Environment variable differences | Dev vs staging vs prod configs | Wrong model loaded, wrong DB |

Docker solves all of these by packaging your **entire runtime environment** into a portable, versioned artifact.

### Images vs Containers vs Registries

- **Image**: A read-only template (like a class definition). Contains the OS, dependencies, code, and model artifacts. Built from a `Dockerfile`.
- **Container**: A running instance of an image (like an object). Isolated process with its own filesystem, network, and process space.
- **Registry**: A repository for storing and distributing images (like PyPI for Python packages). Examples: Docker Hub, AWS ECR, Google GCR.

### Docker Architecture

```
+---------------------------------------------------+
|                  Docker Client (CLI)               |
|  docker build | docker run | docker push           |
+---------------------------------------------------+
          |               |               |
          v               v               v
+---------------------------------------------------+
|                  Docker Daemon                     |
|  Manages images, containers, networks, volumes     |
+---------------------------------------------------+
          |                               |
          v                               v
+-------------------+       +----------------------------+
|   Local Images    |       |    Remote Registry         |
|  (Image Cache)    |       |  (Docker Hub / ECR / GCR)  |
+-------------------+       +----------------------------+
          |
          v
+---------------------------------------------------+
|              Running Containers                    |
|  [API Server]  [Model Worker]  [Redis Cache]       |
+---------------------------------------------------+
```

### Container vs Virtual Machine

| Feature | Container | Virtual Machine |
|---------|-----------|----------------|
| Startup time | Seconds | Minutes |
| Size | MBs (tens to hundreds) | GBs |
| Isolation | Process-level | Full hardware-level |
| OS | Shares host kernel | Full guest OS |
| Performance | Near-native | Hypervisor overhead |
| Density | Dozens per host | A handful per host |
| Best for ML | Serving, CI/CD, microservices | GPU training clusters |

In [ ]:
# Section 1: Check Docker availability

def check_docker():
    """Check if Docker is installed and running."""
    checks = {}
    
    # Check Docker CLI
    try:
        result = subprocess.run(
            ['docker', '--version'],
            capture_output=True, text=True, timeout=10
        )
        checks['docker_cli'] = result.stdout.strip()
        print(f"Docker CLI: {checks['docker_cli']}")
    except (FileNotFoundError, subprocess.TimeoutExpired):
        checks['docker_cli'] = 'Not installed'
        print("Docker CLI: Not installed")
        print("  -> Install from https://docs.docker.com/get-docker/")
    
    # Check Docker Compose
    try:
        result = subprocess.run(
            ['docker', 'compose', 'version'],
            capture_output=True, text=True, timeout=10
        )
        checks['docker_compose'] = result.stdout.strip()
        print(f"Docker Compose: {checks['docker_compose']}")
    except (FileNotFoundError, subprocess.TimeoutExpired):
        checks['docker_compose'] = 'Not available'
        print("Docker Compose: Not available")
    
    # Check if Docker daemon is running
    try:
        result = subprocess.run(
            ['docker', 'info', '--format', '{{.ServerVersion}}'],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            checks['daemon_running'] = True
            print(f"Docker Daemon: Running (Server v{result.stdout.strip()})")
        else:
            checks['daemon_running'] = False
            print("Docker Daemon: Not running (start Docker Desktop)")
    except (FileNotFoundError, subprocess.TimeoutExpired):
        checks['daemon_running'] = False
        print("Docker Daemon: Cannot connect")
    
    return checks

docker_status = check_docker()
print("\nNote: Docker is not required to run this notebook.")
print("We use FastAPI's TestClient for all API testing (no server needed).")

---

## Section 2: Dockerfile Anatomy

A Dockerfile is a text file with instructions for building a Docker image. Each instruction creates a **layer** in the image. Docker caches layers, so ordering matters for build performance.

### Key Instructions

| Instruction | Purpose | Example |
|------------|---------|--------|
| `FROM` | Base image | `FROM python:3.11-slim` |
| `WORKDIR` | Set working directory | `WORKDIR /app` |
| `COPY` | Copy files into image | `COPY requirements.txt .` |
| `RUN` | Execute command during build | `RUN pip install -r requirements.txt` |
| `EXPOSE` | Document port (metadata only) | `EXPOSE 8000` |
| `CMD` | Default command at runtime | `CMD ["uvicorn", "main:app"]` |
| `ENV` | Set environment variable | `ENV MODEL_PATH=/app/model.pkl` |
| `ARG` | Build-time variable | `ARG PYTHON_VERSION=3.11` |
| `HEALTHCHECK` | Container health monitoring | `HEALTHCHECK CMD curl -f http://localhost:8000/health` |

### Layer Ordering Strategy

```
Least frequently changed  -->  Most frequently changed
  OS packages -> Python deps -> Model artifacts -> Application code
```

This maximizes cache hits during rebuilds. If you change application code, Docker only rebuilds from that layer onward.

### Multi-Stage Builds

Multi-stage builds use multiple `FROM` statements. The **builder** stage installs compilation tools and dependencies, while the **runtime** stage copies only the installed packages. This dramatically reduces final image size.

```
Builder Stage (1.2 GB)         Runtime Stage (350 MB)
+--------------------+         +--------------------+
| gcc, build tools   |         | python:3.11-slim   |
| python headers     |  COPY   | site-packages/     |
| pip wheel builds   | ------> | app code           |
| source code        |         | model artifacts    |
+--------------------+         +--------------------+
     (discarded)                  (final image)
```

In [ ]:
# Section 2: Dockerfile for ML model serving

# A production-ready Dockerfile with multi-stage build
dockerfile_content = """\
# =============================================================
# Stage 1: Builder - Install dependencies with build tools
# =============================================================
FROM python:3.11-slim AS builder

# Prevent Python from writing .pyc files and enable unbuffered output
ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1

# Install build dependencies (gcc needed for some ML packages)
RUN apt-get update && \\
    apt-get install -y --no-install-recommends gcc g++ && \\
    rm -rf /var/lib/apt/lists/*

# Create and use a virtual environment
RUN python -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

# Install Python dependencies first (cache-friendly)
COPY requirements.txt .
RUN pip install --no-cache-dir --upgrade pip && \\
    pip install --no-cache-dir -r requirements.txt

# =============================================================
# Stage 2: Runtime - Minimal image for serving
# =============================================================
FROM python:3.11-slim AS runtime

# Security: run as non-root user
RUN groupadd -r mluser && useradd -r -g mluser mluser

ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1

# Copy the virtual environment from builder
COPY --from=builder /opt/venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

# Set the working directory
WORKDIR /app

# Copy model artifacts (changes less often than code)
COPY models/ ./models/

# Copy application code (changes most often - last layer)
COPY app/ ./app/

# Switch to non-root user
USER mluser

# Document the port (metadata only, does not publish)
EXPOSE 8000

# Health check: container orchestrators use this to verify readiness
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

# Start the FastAPI server
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
"""

print("=== Production Dockerfile (Multi-Stage) ===")
print(dockerfile_content)

# Line-by-line explanation
explanations = {
    "FROM python:3.11-slim AS builder": "Start from official Python slim image (Debian-based, ~150MB). 'AS builder' names this stage.",
    "PYTHONDONTWRITEBYTECODE=1": "Prevents .pyc files. Reduces image size, avoids stale bytecode.",
    "PYTHONUNBUFFERED=1": "Forces stdout/stderr to be unbuffered. Essential for seeing logs in real-time.",
    "RUN apt-get ... gcc g++": "Install C compiler for building native Python extensions (numpy wheels, etc.).",
    "python -m venv /opt/venv": "Isolate deps in a venv that we can cleanly copy to the runtime stage.",
    "COPY requirements.txt .": "Copy deps list BEFORE code. If code changes but deps don't, this layer is cached.",
    "COPY --from=builder": "Copy ONLY the installed packages from builder. Build tools are NOT included.",
    "USER mluser": "Security: never run production containers as root. Limits blast radius.",
    "HEALTHCHECK": "Lets Docker/K8s know if the container is healthy. Failed checks trigger restarts.",
    "CMD uvicorn": "Default command. --workers=2 for multi-process serving. Override with docker run.",
}

print("\n=== Key Instruction Explanations ===")
for instruction, explanation in explanations.items():
    print(f"\n  {instruction}")
    print(f"    -> {explanation}")

In [ ]:
# .dockerignore - Files to exclude from the build context

dockerignore_content = """\
# Version control
.git
.gitignore

# Python artifacts
__pycache__
*.pyc
*.pyo
*.egg-info
dist/
build/
.eggs/

# Virtual environments
venv/
.venv/
env/

# IDE / Editor
.vscode/
.idea/
*.swp
*.swo

# Notebooks (not needed in production image)
notebooks/
*.ipynb
.ipynb_checkpoints/

# Data files (too large for image, mount as volumes)
data/
*.csv
*.parquet

# Documentation
*.md
docs/

# Tests (optional - include if you want to test in container)
tests/

# Docker files (avoid recursive context)
Dockerfile*
docker-compose*.yml
.dockerignore

# Secrets (NEVER include these)
.env
.env.*
*.pem
*.key
"""

print("=== .dockerignore ===")
print(dockerignore_content)

print("\n=== Best Practices Summary ===")
best_practices = [
    "Pin exact versions: python:3.11.7-slim, not python:latest",
    "Minimize layers: chain RUN commands with && to reduce image size",
    "Non-root user: always USER <name> before CMD for security",
    "Health checks: HEALTHCHECK lets orchestrators restart unhealthy containers",
    "Multi-stage builds: keep build tools out of the final image",
    "Order by change frequency: deps before code for better caching",
    ".dockerignore: exclude .git, data, notebooks - keep context small",
    "No secrets in images: use env vars or mounted secrets at runtime",
]
for i, practice in enumerate(best_practices, 1):
    print(f"  {i}. {practice}")

---

## Section 3: FastAPI Fundamentals

### Why FastAPI over Flask for ML Serving?

| Feature | FastAPI | Flask |
|---------|---------|-------|
| Performance | Async (ASGI), on par with Node/Go | Sync (WSGI), slower |
| Type hints | Native, drives validation | Manual / no enforcement |
| Auto docs | Swagger UI + ReDoc built-in | Requires flask-restx or flasgger |
| Validation | Pydantic models, automatic | WTForms or marshmallow (manual) |
| Async support | First-class async/await | Requires workarounds |
| Learning curve | Easy if you know Python types | Slightly easier for basics |
| ML ecosystem fit | Great (type safety for model I/O) | Adequate but more boilerplate |

### Path Operations

FastAPI maps HTTP methods to Python functions (called **path operations**):

- `@app.get("/path")` - Read data (health checks, model info)
- `@app.post("/path")` - Send data for processing (predictions)
- `@app.put("/path")` - Update resources (model reloading)
- `@app.delete("/path")` - Remove resources

For ML serving, the two most important are:
- **GET** for health/status/info endpoints
- **POST** for prediction endpoints (sending feature data in the request body)

In [ ]:
# Section 3: Build a simple Calculator API with FastAPI

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

# Create the FastAPI application
app = FastAPI(
    title="Calculator API",
    description="A simple calculator to demonstrate FastAPI fundamentals",
    version="1.0.0"
)

# Request model - Pydantic validates this automatically
class CalcRequest(BaseModel):
    a: float
    b: float
    operation: str  # add, subtract, multiply, divide

# Response model - documents the API output
class CalcResponse(BaseModel):
    result: float
    operation: str
    expression: str

# GET endpoint - simple root
@app.get("/")
def root():
    return {"message": "Calculator API is running", "version": "1.0.0"}

# POST endpoint - perform calculation
@app.post("/calculate", response_model=CalcResponse)
def calculate(req: CalcRequest):
    """Perform a mathematical operation on two numbers."""
    operations = {
        "add": lambda a, b: a + b,
        "subtract": lambda a, b: a - b,
        "multiply": lambda a, b: a * b,
        "divide": lambda a, b: a / b if b != 0 else None,
    }
    
    if req.operation not in operations:
        raise HTTPException(
            status_code=400,
            detail=f"Unknown operation '{req.operation}'. Use: {list(operations.keys())}"
        )
    
    result = operations[req.operation](req.a, req.b)
    
    if result is None:
        raise HTTPException(status_code=400, detail="Division by zero")
    
    symbols = {"add": "+", "subtract": "-", "multiply": "*", "divide": "/"}
    
    return CalcResponse(
        result=result,
        operation=req.operation,
        expression=f"{req.a} {symbols[req.operation]} {req.b} = {result}"
    )

print("Calculator API created. Testing with TestClient...")
print("(No server needed - TestClient calls the app directly)\n")

In [ ]:
# Test the Calculator API with TestClient
client = TestClient(app)

# Test GET /
response = client.get("/")
print(f"GET / -> {response.status_code}: {response.json()}")

# Test calculations
test_cases = [
    {"a": 10, "b": 5, "operation": "add"},
    {"a": 10, "b": 5, "operation": "subtract"},
    {"a": 10, "b": 5, "operation": "multiply"},
    {"a": 10, "b": 5, "operation": "divide"},
]

print("\n--- Calculation Results ---")
for tc in test_cases:
    resp = client.post("/calculate", json=tc)
    data = resp.json()
    print(f"  {data['expression']}")

# Test error handling
print("\n--- Error Handling ---")

# Invalid operation
resp = client.post("/calculate", json={"a": 1, "b": 2, "operation": "power"})
print(f"  Invalid operation -> {resp.status_code}: {resp.json()['detail']}")

# Division by zero
resp = client.post("/calculate", json={"a": 10, "b": 0, "operation": "divide"})
print(f"  Divide by zero   -> {resp.status_code}: {resp.json()['detail']}")

# Missing field (Pydantic validation)
resp = client.post("/calculate", json={"a": 10, "operation": "add"})
print(f"  Missing field 'b' -> {resp.status_code}: {resp.json()['detail'][0]['msg']}")

---

## Section 4: Pydantic for Input Validation

Pydantic is the backbone of FastAPI's validation system. For ML model serving, robust input validation is critical because:

1. **Garbage in, garbage out** - invalid features produce meaningless predictions
2. **Security** - malicious inputs can crash your model or exploit vulnerabilities
3. **Debugging** - clear validation errors are easier to trace than model exceptions
4. **Documentation** - Pydantic models auto-generate OpenAPI schemas

### Key Pydantic V2 Features

- **Field constraints**: `ge`, `le`, `gt`, `lt`, `min_length`, `max_length`, `pattern`
- **Custom validators**: `@field_validator` for complex logic
- **Nested models**: Models referencing other models
- **JSON schema**: Auto-generated from model definitions
- **Config**: `model_config` for customization and examples

In [ ]:
# Section 4: Pydantic models for churn prediction

from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Optional, List
from enum import Enum


class ContractType(str, Enum):
    """Valid contract types for the churn model."""
    MONTH_TO_MONTH = "Month-to-month"
    ONE_YEAR = "One year"
    TWO_YEAR = "Two year"


class PaymentMethod(str, Enum):
    ELECTRONIC_CHECK = "Electronic check"
    MAILED_CHECK = "Mailed check"
    BANK_TRANSFER = "Bank transfer"
    CREDIT_CARD = "Credit card"


class CustomerFeatures(BaseModel):
    """Input features for churn prediction model.
    
    All fields are validated against realistic business constraints.
    """
    # Numeric features with range constraints
    tenure_months: int = Field(
        ge=0, le=120,
        description="Number of months the customer has been with the company"
    )
    monthly_charges: float = Field(
        ge=0.0, le=500.0,
        description="Monthly charge amount in dollars"
    )
    total_charges: float = Field(
        ge=0.0,
        description="Total charges over the customer's lifetime"
    )
    
    # Categorical features using Enum validation
    contract: ContractType = Field(
        description="Type of contract"
    )
    payment_method: PaymentMethod = Field(
        description="Payment method used"
    )
    
    # Boolean features
    has_internet_service: bool = Field(
        description="Whether customer has internet service"
    )
    has_tech_support: bool = Field(
        default=False,
        description="Whether customer has tech support add-on"
    )
    
    # Optional features
    num_dependents: Optional[int] = Field(
        default=0, ge=0, le=10,
        description="Number of dependents on the account"
    )
    
    # Custom validator: total_charges should be >= tenure * monthly_charges * 0.5
    @field_validator('total_charges')
    @classmethod
    def validate_total_charges(cls, v, info):
        """Total charges should be non-negative."""
        if v < 0:
            raise ValueError('Total charges cannot be negative')
        return v
    
    # Cross-field validation
    @model_validator(mode='after')
    def validate_charges_consistency(self):
        """Total charges should be at least one month of monthly charges."""
        if self.tenure_months > 0 and self.total_charges < self.monthly_charges:
            raise ValueError(
                f'Total charges ({self.total_charges}) cannot be less than '
                f'monthly charges ({self.monthly_charges}) for tenure > 0'
            )
        return self
    
    # Provide example for OpenAPI docs
    model_config = {
        "json_schema_extra": {
            "examples": [
                {
                    "tenure_months": 24,
                    "monthly_charges": 79.99,
                    "total_charges": 1919.76,
                    "contract": "One year",
                    "payment_method": "Credit card",
                    "has_internet_service": True,
                    "has_tech_support": True,
                    "num_dependents": 2
                }
            ]
        }
    }


# Nested model for prediction responses
class PredictionResponse(BaseModel):
    """Response from the churn prediction endpoint."""
    prediction: str = Field(description="'churn' or 'no_churn'")
    probability: float = Field(ge=0.0, le=1.0, description="Churn probability")
    confidence: str = Field(description="'low', 'medium', or 'high'")
    model_version: str = Field(description="Version of the model used")


print("=== CustomerFeatures Schema ===")
print(json.dumps(CustomerFeatures.model_json_schema(), indent=2))

In [ ]:
# Demonstrate validation with valid and invalid inputs

print("=== Valid Input ===")
valid_customer = CustomerFeatures(
    tenure_months=24,
    monthly_charges=79.99,
    total_charges=1919.76,
    contract=ContractType.ONE_YEAR,
    payment_method=PaymentMethod.CREDIT_CARD,
    has_internet_service=True,
    has_tech_support=True,
    num_dependents=2
)
print(f"  Created: {valid_customer.model_dump()}")

# Invalid inputs - demonstrate validation errors
print("\n=== Validation Errors ===")

invalid_cases = [
    {
        "name": "Negative tenure",
        "data": {"tenure_months": -5, "monthly_charges": 50, "total_charges": 600,
                 "contract": "One year", "payment_method": "Credit card",
                 "has_internet_service": True}
    },
    {
        "name": "Charges too high",
        "data": {"tenure_months": 10, "monthly_charges": 999, "total_charges": 9990,
                 "contract": "One year", "payment_method": "Credit card",
                 "has_internet_service": True}
    },
    {
        "name": "Invalid contract type",
        "data": {"tenure_months": 10, "monthly_charges": 50, "total_charges": 500,
                 "contract": "Lifetime", "payment_method": "Credit card",
                 "has_internet_service": True}
    },
    {
        "name": "Total < monthly (cross-field)",
        "data": {"tenure_months": 12, "monthly_charges": 100, "total_charges": 50,
                 "contract": "One year", "payment_method": "Credit card",
                 "has_internet_service": True}
    },
]

for case in invalid_cases:
    try:
        CustomerFeatures(**case["data"])
        print(f"  {case['name']}: PASSED (unexpected)")
    except Exception as e:
        error_msg = str(e).split('\n')[1] if '\n' in str(e) else str(e)[:100]
        print(f"  {case['name']}: CAUGHT -> {error_msg}")

---

## Section 5: Serving an ML Model

Now we combine everything: train a model and serve it through a FastAPI application with proper request/response validation.

### Model Loading Patterns

1. **Global variable** (simple): Load model at module import time
   ```python
   model = joblib.load("model.pkl")  # loaded when module imports
   ```

2. **Lifespan event** (recommended): Load during app startup, cleanup on shutdown
   ```python
   @asynccontextmanager
   async def lifespan(app: FastAPI):
       app.state.model = joblib.load("model.pkl")  # startup
       yield
       del app.state.model  # shutdown cleanup
   ```

3. **Lazy loading** (conditional): Load on first request
   ```python
   _model = None
   def get_model():
       global _model
       if _model is None:
           _model = joblib.load("model.pkl")
       return _model
   ```

### Endpoint Design

- **`/predict`** - Single prediction (real-time, low latency)
- **`/predict/batch`** - Batch predictions (higher throughput for multiple items)

In [ ]:
# Section 5: Train a model on synthetic churn data

np.random.seed(42)
n_samples = 2000

# Generate synthetic churn data
data = pd.DataFrame({
    'tenure_months': np.random.randint(0, 72, n_samples),
    'monthly_charges': np.random.uniform(20, 120, n_samples).round(2),
    'contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.5, 0.3, 0.2]),
    'payment_method': np.random.choice(
        ['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n_samples
    ),
    'has_internet_service': np.random.choice([0, 1], n_samples, p=[0.3, 0.7]),
    'has_tech_support': np.random.choice([0, 1], n_samples, p=[0.6, 0.4]),
    'num_dependents': np.random.randint(0, 5, n_samples),
})

# Calculate total_charges based on tenure and monthly
data['total_charges'] = (data['tenure_months'] * data['monthly_charges'] * 
                         np.random.uniform(0.85, 1.15, n_samples)).round(2)

# Generate churn labels (correlated with features)
churn_score = (
    -0.03 * data['tenure_months'] +
    0.01 * data['monthly_charges'] +
    0.8 * (data['contract'] == 'Month-to-month').astype(int) +
    0.3 * (data['payment_method'] == 'Electronic check').astype(int) -
    0.3 * data['has_tech_support'] -
    0.1 * data['num_dependents'] +
    np.random.normal(0, 0.5, n_samples)
)
data['churn'] = (churn_score > np.median(churn_score)).astype(int)

print(f"Dataset shape: {data.shape}")
print(f"Churn rate: {data['churn'].mean():.1%}")
print(f"\nFeature sample:")
data.head(3)

In [ ]:
# Encode categoricals and train GBM model

# Encode categorical features
contract_encoder = LabelEncoder()
payment_encoder = LabelEncoder()

data['contract_encoded'] = contract_encoder.fit_transform(data['contract'])
data['payment_encoded'] = payment_encoder.fit_transform(data['payment_method'])

feature_columns = [
    'tenure_months', 'monthly_charges', 'total_charges',
    'contract_encoded', 'payment_encoded',
    'has_internet_service', 'has_tech_support', 'num_dependents'
]

X = data[feature_columns].values
y = data['churn'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Gradient Boosting Classifier
model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate
train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))

print(f"Training accuracy: {train_acc:.4f}")
print(f"Test accuracy:     {test_acc:.4f}")
print(f"\nClassification Report (Test Set):")
print(classification_report(y_test, model.predict(X_test), target_names=['No Churn', 'Churn']))

# Store model metadata
MODEL_VERSION = "1.0.0"
MODEL_TRAINED_AT = datetime.now().isoformat()
print(f"\nModel version: {MODEL_VERSION}")
print(f"Trained at: {MODEL_TRAINED_AT}")

In [ ]:
# Build the ML Serving API

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List

# --- Pydantic Models ---

class ChurnFeatures(BaseModel):
    """Input features for churn prediction."""
    tenure_months: int = Field(ge=0, le=120)
    monthly_charges: float = Field(ge=0, le=500)
    total_charges: float = Field(ge=0)
    contract: str = Field(description="Month-to-month, One year, or Two year")
    payment_method: str = Field(description="Payment method")
    has_internet_service: bool
    has_tech_support: bool = False
    num_dependents: int = Field(default=0, ge=0, le=10)

class ChurnPrediction(BaseModel):
    """Single prediction response."""
    prediction: str
    churn_probability: float = Field(ge=0.0, le=1.0)
    confidence: str
    model_version: str

class BatchRequest(BaseModel):
    """Batch prediction request."""
    customers: List[ChurnFeatures]

class BatchResponse(BaseModel):
    """Batch prediction response."""
    predictions: List[ChurnPrediction]
    count: int
    processing_time_ms: float

# --- Helper Functions ---

def encode_features(features: ChurnFeatures) -> np.ndarray:
    """Convert Pydantic model to numpy array for model input."""
    # Encode contract type
    contract_map = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
    payment_map = {'Bank transfer': 0, 'Credit card': 1, 'Electronic check': 2, 'Mailed check': 3}
    
    if features.contract not in contract_map:
        raise HTTPException(status_code=400, detail=f"Invalid contract: {features.contract}")
    if features.payment_method not in payment_map:
        raise HTTPException(status_code=400, detail=f"Invalid payment method: {features.payment_method}")
    
    return np.array([[
        features.tenure_months,
        features.monthly_charges,
        features.total_charges,
        contract_map[features.contract],
        payment_map[features.payment_method],
        int(features.has_internet_service),
        int(features.has_tech_support),
        features.num_dependents
    ]])

def make_prediction(features: ChurnFeatures) -> ChurnPrediction:
    """Generate a prediction for a single customer."""
    X = encode_features(features)
    proba = model.predict_proba(X)[0]
    churn_prob = float(proba[1])
    
    # Determine confidence
    if churn_prob > 0.8 or churn_prob < 0.2:
        confidence = "high"
    elif churn_prob > 0.65 or churn_prob < 0.35:
        confidence = "medium"
    else:
        confidence = "low"
    
    return ChurnPrediction(
        prediction="churn" if churn_prob >= 0.5 else "no_churn",
        churn_probability=round(churn_prob, 4),
        confidence=confidence,
        model_version=MODEL_VERSION
    )

# --- FastAPI App ---

ml_app = FastAPI(
    title="Churn Prediction API",
    description="Predict customer churn using Gradient Boosting",
    version=MODEL_VERSION
)

@ml_app.get("/")
def root():
    return {"service": "Churn Prediction API", "version": MODEL_VERSION}

@ml_app.post("/predict", response_model=ChurnPrediction)
def predict(features: ChurnFeatures):
    """Predict churn for a single customer."""
    return make_prediction(features)

@ml_app.post("/predict/batch", response_model=BatchResponse)
def predict_batch(batch: BatchRequest):
    """Predict churn for multiple customers."""
    if len(batch.customers) > 100:
        raise HTTPException(status_code=400, detail="Maximum batch size is 100")
    
    start_time = time.time()
    predictions = [make_prediction(customer) for customer in batch.customers]
    elapsed_ms = (time.time() - start_time) * 1000
    
    return BatchResponse(
        predictions=predictions,
        count=len(predictions),
        processing_time_ms=round(elapsed_ms, 2)
    )

print("ML Serving API created with /predict and /predict/batch endpoints.")

In [ ]:
# Test the ML Serving API end-to-end

ml_client = TestClient(ml_app)

# Single prediction
print("=== Single Prediction ===")
customer_data = {
    "tenure_months": 3,
    "monthly_charges": 89.99,
    "total_charges": 269.97,
    "contract": "Month-to-month",
    "payment_method": "Electronic check",
    "has_internet_service": True,
    "has_tech_support": False,
    "num_dependents": 0
}

resp = ml_client.post("/predict", json=customer_data)
print(f"Status: {resp.status_code}")
print(f"Response: {json.dumps(resp.json(), indent=2)}")

# A loyal customer - should be low churn risk
print("\n=== Loyal Customer ===")
loyal_customer = {
    "tenure_months": 60,
    "monthly_charges": 45.00,
    "total_charges": 2700.00,
    "contract": "Two year",
    "payment_method": "Credit card",
    "has_internet_service": True,
    "has_tech_support": True,
    "num_dependents": 3
}

resp = ml_client.post("/predict", json=loyal_customer)
print(f"Response: {json.dumps(resp.json(), indent=2)}")

# Batch prediction
print("\n=== Batch Prediction ===")
batch_data = {
    "customers": [customer_data, loyal_customer, {
        "tenure_months": 12,
        "monthly_charges": 70.00,
        "total_charges": 840.00,
        "contract": "One year",
        "payment_method": "Bank transfer",
        "has_internet_service": True,
        "has_tech_support": False,
        "num_dependents": 1
    }]
}

resp = ml_client.post("/predict/batch", json=batch_data)
result = resp.json()
print(f"Predictions: {result['count']}")
print(f"Processing time: {result['processing_time_ms']:.2f} ms")
for i, pred in enumerate(result['predictions']):
    print(f"  Customer {i+1}: {pred['prediction']} (prob={pred['churn_probability']}, confidence={pred['confidence']})")

---

## Section 6: Middleware and Logging

Middleware sits between the client and your endpoint logic. Every request passes through middleware in order, and every response passes back through in reverse order.

```
Client Request
    |                          Middleware Stack
    v                    +------------------------+
  [CORS Middleware]  --> | Check origin headers   |
    |                    +------------------------+
    v                    +------------------------+
  [Timing Middleware] -> | Start timer            |
    |                    +------------------------+
    v                    +------------------------+
  [Logging Middleware] ->| Log request details    |
    |                    +------------------------+
    v
  [Endpoint Handler]  -- process request, return response
    |
    v (response flows back up)
  [Logging] -> log response status
  [Timing]  -> add X-Process-Time header
  [CORS]    -> add Access-Control headers
    |
    v
Client Response
```

### Why Middleware Matters for ML APIs

- **Timing**: Track inference latency (SLA monitoring)
- **Logging**: Audit trail for predictions (regulatory compliance)
- **Error handling**: Graceful degradation, not stack traces to clients
- **CORS**: Allow browser-based dashboards to call your API
- **Request ID**: Trace a single request across distributed services

In [ ]:
# Section 6: Build an API with full middleware stack

import uuid
import logging
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from starlette.middleware.base import BaseHTTPMiddleware

# Configure structured logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ml_api")


# --- Custom Middleware Classes ---

class TimingMiddleware(BaseHTTPMiddleware):
    """Add processing time to response headers."""
    async def dispatch(self, request: Request, call_next):
        start_time = time.time()
        response = await call_next(request)
        process_time = (time.time() - start_time) * 1000  # ms
        response.headers["X-Process-Time-Ms"] = f"{process_time:.2f}"
        return response


class RequestIDMiddleware(BaseHTTPMiddleware):
    """Add a unique request ID to every request/response."""
    async def dispatch(self, request: Request, call_next):
        request_id = request.headers.get("X-Request-ID", str(uuid.uuid4())[:8])
        request.state.request_id = request_id
        response = await call_next(request)
        response.headers["X-Request-ID"] = request_id
        return response


class LoggingMiddleware(BaseHTTPMiddleware):
    """Log every request and response in structured format."""
    async def dispatch(self, request: Request, call_next):
        # Log request
        request_id = getattr(request.state, 'request_id', 'unknown')
        log_data = {
            "request_id": request_id,
            "method": request.method,
            "path": request.url.path,
            "client": request.client.host if request.client else "unknown",
        }
        logger.info(f"REQUEST: {json.dumps(log_data)}")
        
        response = await call_next(request)
        
        # Log response
        log_data["status_code"] = response.status_code
        log_data["process_time_ms"] = response.headers.get("X-Process-Time-Ms", "N/A")
        logger.info(f"RESPONSE: {json.dumps(log_data)}")
        
        return response


# --- Build the App with Middleware ---

middleware_app = FastAPI(title="ML API with Middleware")

# Add middleware (order matters - first added = outermost)
middleware_app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000", "https://dashboard.example.com"],
    allow_credentials=True,
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)
middleware_app.add_middleware(LoggingMiddleware)
middleware_app.add_middleware(TimingMiddleware)
middleware_app.add_middleware(RequestIDMiddleware)


# Custom exception handler
class ModelPredictionError(Exception):
    def __init__(self, message: str, request_id: str = None):
        self.message = message
        self.request_id = request_id

@middleware_app.exception_handler(ModelPredictionError)
async def model_error_handler(request: Request, exc: ModelPredictionError):
    """Return structured error response instead of stack trace."""
    return JSONResponse(
        status_code=500,
        content={
            "error": "model_prediction_error",
            "message": exc.message,
            "request_id": exc.request_id or "unknown",
        }
    )

@middleware_app.exception_handler(Exception)
async def generic_error_handler(request: Request, exc: Exception):
    """Catch-all handler: never expose internal errors to clients."""
    logger.error(f"Unhandled error: {type(exc).__name__}: {exc}")
    return JSONResponse(
        status_code=500,
        content={
            "error": "internal_server_error",
            "message": "An unexpected error occurred. Please try again.",
        }
    )


# Endpoints
@middleware_app.get("/")
def root():
    return {"service": "ML API with Middleware", "status": "running"}

@middleware_app.post("/predict")
def predict(features: ChurnFeatures):
    return make_prediction(features)


print("Middleware API created with:")
print("  - CORS middleware")
print("  - Request timing (X-Process-Time-Ms header)")
print("  - Request ID tracking (X-Request-ID header)")
print("  - Structured JSON logging")
print("  - Custom exception handlers")

In [ ]:
# Test middleware behavior

mw_client = TestClient(middleware_app)

print("=== Middleware in Action ===")

# Normal request - observe headers
resp = mw_client.get("/")
print(f"\n--- GET / ---")
print(f"Status: {resp.status_code}")
print(f"Body: {resp.json()}")
print(f"X-Process-Time-Ms: {resp.headers.get('X-Process-Time-Ms', 'N/A')}")
print(f"X-Request-ID: {resp.headers.get('X-Request-ID', 'N/A')}")

# Prediction request with custom request ID
resp = mw_client.post(
    "/predict",
    json={
        "tenure_months": 6,
        "monthly_charges": 95.00,
        "total_charges": 570.00,
        "contract": "Month-to-month",
        "payment_method": "Electronic check",
        "has_internet_service": True,
        "has_tech_support": False,
        "num_dependents": 0
    },
    headers={"X-Request-ID": "test-req-001"}
)
print(f"\n--- POST /predict (with custom request ID) ---")
print(f"Status: {resp.status_code}")
print(f"Prediction: {resp.json()['prediction']}")
print(f"X-Process-Time-Ms: {resp.headers.get('X-Process-Time-Ms', 'N/A')}")
print(f"X-Request-ID: {resp.headers.get('X-Request-ID', 'N/A')}")

# Validation error - middleware still runs
resp = mw_client.post("/predict", json={"tenure_months": -1})
print(f"\n--- POST /predict (invalid input) ---")
print(f"Status: {resp.status_code}")
print(f"X-Request-ID: {resp.headers.get('X-Request-ID', 'N/A')}")
print(f"Error: {resp.json()['detail'][0]['msg']}")

---

## Section 7: Health Checks and Metrics

Production ML services need observability endpoints that container orchestrators (Kubernetes, ECS) and monitoring systems (Prometheus, Datadog) can query.

### Key Endpoints

| Endpoint | Purpose | Consumer |
|----------|---------|----------|
| `/health` | Is the service alive and ready? | Kubernetes liveness/readiness probes |
| `/metrics` | Request counts, latencies, error rates | Prometheus scraper |
| `/model/info` | Model version, features, training date | DevOps dashboards, debugging |

### Kubernetes Probes

```yaml
livenessProbe:       # Is the container running?
  httpGet:
    path: /health
    port: 8000
  periodSeconds: 30

readinessProbe:      # Is the container ready to serve traffic?
  httpGet:
    path: /health
    port: 8000
  initialDelaySeconds: 10
```

If `/health` returns a non-200 status, Kubernetes will:
- **Liveness failure**: Restart the container
- **Readiness failure**: Remove from load balancer (stop sending traffic)

In [ ]:
# Section 7: Health checks and metrics API

from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Dict, Any

# --- Response Models ---

class HealthResponse(BaseModel):
    """Health check response for orchestration probes."""
    status: str = Field(description="'healthy' or 'unhealthy'")
    model_loaded: bool = Field(description="Whether the ML model is loaded")
    uptime_seconds: float = Field(description="Seconds since service started")
    version: str = Field(description="API version")

class MetricsResponse(BaseModel):
    """Prometheus-style metrics."""
    total_requests: int
    total_predictions: int
    total_errors: int
    avg_latency_ms: float
    model_version: str
    uptime_seconds: float

class ModelInfoResponse(BaseModel):
    """Detailed model information."""
    model_type: str
    version: str
    trained_at: str
    features: list
    n_estimators: int
    test_accuracy: float


# --- App with Observability ---

# In-memory metrics store (use Redis/Prometheus in production)
metrics_store = {
    "start_time": time.time(),
    "total_requests": 0,
    "total_predictions": 0,
    "total_errors": 0,
    "latencies": [],
}

health_app = FastAPI(title="ML API with Health Checks")


@health_app.get("/health", response_model=HealthResponse)
def health_check():
    """Health check endpoint for Kubernetes probes.
    
    Returns 200 if healthy, 503 if the model is not loaded.
    """
    metrics_store["total_requests"] += 1
    
    model_loaded = model is not None
    status = "healthy" if model_loaded else "unhealthy"
    uptime = time.time() - metrics_store["start_time"]
    
    response = HealthResponse(
        status=status,
        model_loaded=model_loaded,
        uptime_seconds=round(uptime, 2),
        version=MODEL_VERSION
    )
    
    if not model_loaded:
        raise HTTPException(status_code=503, detail=response.model_dump())
    
    return response


@health_app.get("/metrics", response_model=MetricsResponse)
def get_metrics():
    """Metrics endpoint for Prometheus scraping."""
    metrics_store["total_requests"] += 1
    uptime = time.time() - metrics_store["start_time"]
    
    latencies = metrics_store["latencies"]
    avg_latency = sum(latencies) / len(latencies) if latencies else 0.0
    
    return MetricsResponse(
        total_requests=metrics_store["total_requests"],
        total_predictions=metrics_store["total_predictions"],
        total_errors=metrics_store["total_errors"],
        avg_latency_ms=round(avg_latency, 2),
        model_version=MODEL_VERSION,
        uptime_seconds=round(uptime, 2)
    )


@health_app.get("/model/info", response_model=ModelInfoResponse)
def model_info():
    """Return detailed model information."""
    metrics_store["total_requests"] += 1
    
    return ModelInfoResponse(
        model_type="GradientBoostingClassifier",
        version=MODEL_VERSION,
        trained_at=MODEL_TRAINED_AT,
        features=feature_columns,
        n_estimators=model.n_estimators,
        test_accuracy=round(test_acc, 4)
    )


@health_app.post("/predict")
def predict_with_metrics(features: ChurnFeatures):
    """Predict with metrics tracking."""
    metrics_store["total_requests"] += 1
    
    start = time.time()
    try:
        result = make_prediction(features)
        latency_ms = (time.time() - start) * 1000
        metrics_store["total_predictions"] += 1
        metrics_store["latencies"].append(latency_ms)
        # Keep only last 1000 latencies
        if len(metrics_store["latencies"]) > 1000:
            metrics_store["latencies"] = metrics_store["latencies"][-1000:]
        return result
    except Exception as e:
        metrics_store["total_errors"] += 1
        raise


print("Health & Metrics API created.")
print("Endpoints: /health, /metrics, /model/info, /predict")

In [ ]:
# Test health checks and metrics

health_client = TestClient(health_app)

# Health check
print("=== Health Check ===")
resp = health_client.get("/health")
print(json.dumps(resp.json(), indent=2))

# Model info
print("\n=== Model Info ===")
resp = health_client.get("/model/info")
print(json.dumps(resp.json(), indent=2))

# Make several predictions to build up metrics
print("\n=== Making 5 predictions... ===")
for i in range(5):
    resp = health_client.post("/predict", json={
        "tenure_months": np.random.randint(1, 60),
        "monthly_charges": round(np.random.uniform(30, 100), 2),
        "total_charges": round(np.random.uniform(100, 5000), 2),
        "contract": np.random.choice(["Month-to-month", "One year", "Two year"]),
        "payment_method": np.random.choice(["Electronic check", "Credit card", "Bank transfer", "Mailed check"]),
        "has_internet_service": bool(np.random.choice([True, False])),
        "has_tech_support": bool(np.random.choice([True, False])),
        "num_dependents": int(np.random.randint(0, 4))
    })
    pred = resp.json()
    print(f"  Prediction {i+1}: {pred['prediction']} (p={pred['churn_probability']})")

# Check metrics after predictions
print("\n=== Metrics After Predictions ===")
resp = health_client.get("/metrics")
print(json.dumps(resp.json(), indent=2))

---

## Section 8: Docker Compose

Docker Compose orchestrates multiple containers as a single application. For ML services, a typical setup includes:

```
+------------------------------------------------------+
|                   Docker Compose                      |
|                                                      |
|  +-----------+    +-------------+    +------------+  |
|  |  Nginx    | -> |  FastAPI    | -> |  Redis     |  |
|  |  (proxy)  |    |  (ML API)   |    |  (cache)   |  |
|  |  :80      |    |  :8000      |    |  :6379     |  |
|  +-----------+    +-------------+    +------------+  |
|                         |                            |
|                   +-----v------+                     |
|                   |  Volume:   |                     |
|                   |  models/   |                     |
|                   +------------+                     |
+------------------------------------------------------+
```

### Key Compose Concepts

- **Services**: Each container definition (api, nginx, redis)
- **Networks**: Internal DNS for service-to-service communication
- **Volumes**: Persistent storage that survives container restarts
- **Environment variables**: Configuration injection without rebuilding images
- **Depends on**: Startup ordering between services
- **Health checks**: Wait for dependencies to be ready

In [ ]:
# Section 8: Docker Compose configuration

docker_compose_content = """\
# docker-compose.yml - ML Model Serving Stack
# Usage: docker compose up --build

version: '3.8'

services:
  # ============================================
  # ML API Service - The core prediction server
  # ============================================
  api:
    build:
      context: .                    # Build from current directory
      dockerfile: Dockerfile        # Use our multi-stage Dockerfile
    container_name: churn-api
    ports:
      - "8000:8000"                 # Map host:container port
    environment:
      - MODEL_PATH=/app/models/churn_model.pkl
      - LOG_LEVEL=INFO
      - MAX_BATCH_SIZE=100
      - WORKERS=2
    volumes:
      - model-store:/app/models     # Persist models across restarts
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 15s             # Grace period for model loading
    restart: unless-stopped          # Auto-restart on failure
    networks:
      - ml-network
    deploy:
      resources:
        limits:
          memory: 2G                 # Memory limit for the container
          cpus: '2.0'               # CPU limit

  # ============================================
  # Nginx Reverse Proxy - SSL termination & load balancing
  # ============================================
  nginx:
    image: nginx:1.25-alpine
    container_name: churn-proxy
    ports:
      - "80:80"                      # HTTP
      - "443:443"                    # HTTPS
    volumes:
      - ./nginx/nginx.conf:/etc/nginx/nginx.conf:ro   # Read-only config
      - ./nginx/certs:/etc/nginx/certs:ro             # SSL certificates
    depends_on:
      api:
        condition: service_healthy   # Wait for API health check
    networks:
      - ml-network
    restart: unless-stopped

  # ============================================
  # Redis - Optional caching for predictions
  # ============================================
  redis:
    image: redis:7-alpine
    container_name: churn-cache
    ports:
      - "6379:6379"
    volumes:
      - redis-data:/data             # Persist cache across restarts
    command: redis-server --maxmemory 256mb --maxmemory-policy allkeys-lru
    networks:
      - ml-network
    restart: unless-stopped

# Named volumes - managed by Docker
volumes:
  model-store:
    driver: local
  redis-data:
    driver: local

# Internal network for service communication
networks:
  ml-network:
    driver: bridge
"""

print("=== docker-compose.yml ===")
print(docker_compose_content)

In [ ]:
# Nginx configuration for reverse proxy

nginx_conf = """\
# nginx/nginx.conf - Reverse proxy for ML API

upstream ml_api {
    server api:8000;          # Docker Compose DNS resolves 'api' to the container
    # Add more servers here for load balancing:
    # server api-2:8000;
    # server api-3:8000;
}

server {
    listen 80;
    server_name _;            # Accept any hostname

    # Rate limiting zone
    limit_req_zone $binary_remote_addr zone=api_limit:10m rate=10r/s;

    location / {
        proxy_pass http://ml_api;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Request-ID $request_id;

        # Timeouts
        proxy_connect_timeout 10s;
        proxy_read_timeout 30s;    # Allow up to 30s for batch predictions
        proxy_send_timeout 10s;

        # Rate limiting
        limit_req zone=api_limit burst=20 nodelay;
    }

    # Health check bypass (no rate limiting)
    location /health {
        proxy_pass http://ml_api/health;
    }

    # Metrics bypass
    location /metrics {
        proxy_pass http://ml_api/metrics;
        allow 10.0.0.0/8;          # Only internal network
        deny all;
    }
}
"""

print("=== nginx.conf ===")
print(nginx_conf)

print("\n=== Docker Compose Section Explanations ===")
explanations = [
    ("services.api.build", "Builds from local Dockerfile. 'context: .' sends the current dir as build context."),
    ("services.api.environment", "Inject config at runtime. Never hardcode paths, credentials, or settings."),
    ("services.api.volumes", "Named volume 'model-store' persists model files across container restarts."),
    ("services.api.healthcheck", "Compose checks /health every 30s. 3 failures -> mark unhealthy."),
    ("services.api.deploy.resources", "Memory/CPU limits prevent one service from starving others."),
    ("services.nginx.depends_on", "condition: service_healthy waits for the API to pass its health check."),
    ("services.redis.command", "LRU eviction with 256MB cap. Old predictions get evicted automatically."),
    ("networks.ml-network", "Isolated bridge network. Services resolve each other by name (e.g., 'api:8000')."),
]
for section, explanation in explanations:
    print(f"\n  {section}:")
    print(f"    {explanation}")

---

## Section 9: Testing APIs with TestClient

FastAPI's `TestClient` (powered by httpx) lets you test your API **without starting a server**. This is critical for:

- **CI/CD pipelines**: Tests run in containers without exposed ports
- **Speed**: No server startup overhead
- **Isolation**: Each test gets a fresh client
- **Coverage**: Test validation, edge cases, and error handling

### Test Categories

1. **Happy path**: Valid inputs produce expected outputs
2. **Validation errors**: Invalid inputs return 422 with clear messages
3. **Edge cases**: Boundary values, empty inputs, maximum batch sizes
4. **Error handling**: Server errors return structured responses, not stack traces
5. **Performance**: Response time within SLA bounds

In [ ]:
# Section 9: Comprehensive test suite for the ML API

from fastapi.testclient import TestClient

# Use the ml_app from Section 5 (has /predict and /predict/batch)
test_client = TestClient(ml_app)

# Track test results
test_results = []

def run_test(name: str, test_func):
    """Run a test and record pass/fail."""
    try:
        test_func()
        test_results.append((name, "PASS"))
        print(f"  PASS: {name}")
    except AssertionError as e:
        test_results.append((name, f"FAIL: {e}"))
        print(f"  FAIL: {name} -> {e}")
    except Exception as e:
        test_results.append((name, f"ERROR: {e}"))
        print(f"  ERROR: {name} -> {e}")


# --- Happy Path Tests ---

print("=== Happy Path Tests ===")

def test_root():
    resp = test_client.get("/")
    assert resp.status_code == 200
    assert "service" in resp.json()

def test_predict_valid():
    resp = test_client.post("/predict", json={
        "tenure_months": 24,
        "monthly_charges": 79.99,
        "total_charges": 1919.76,
        "contract": "One year",
        "payment_method": "Credit card",
        "has_internet_service": True,
        "has_tech_support": True,
        "num_dependents": 2
    })
    assert resp.status_code == 200
    data = resp.json()
    assert data["prediction"] in ["churn", "no_churn"]
    assert 0 <= data["churn_probability"] <= 1
    assert data["confidence"] in ["low", "medium", "high"]
    assert data["model_version"] == MODEL_VERSION

def test_predict_high_risk_customer():
    """Month-to-month + electronic check + low tenure = high churn risk."""
    resp = test_client.post("/predict", json={
        "tenure_months": 2,
        "monthly_charges": 99.99,
        "total_charges": 199.98,
        "contract": "Month-to-month",
        "payment_method": "Electronic check",
        "has_internet_service": True,
        "has_tech_support": False,
        "num_dependents": 0
    })
    assert resp.status_code == 200
    # High risk customer should have elevated churn probability
    assert resp.json()["churn_probability"] > 0.3

def test_predict_low_risk_customer():
    """Two year contract + long tenure + tech support = low churn risk."""
    resp = test_client.post("/predict", json={
        "tenure_months": 60,
        "monthly_charges": 40.00,
        "total_charges": 2400.00,
        "contract": "Two year",
        "payment_method": "Credit card",
        "has_internet_service": True,
        "has_tech_support": True,
        "num_dependents": 3
    })
    assert resp.status_code == 200
    assert resp.json()["churn_probability"] < 0.7

run_test("GET /", test_root)
run_test("POST /predict (valid input)", test_predict_valid)
run_test("POST /predict (high risk customer)", test_predict_high_risk_customer)
run_test("POST /predict (low risk customer)", test_predict_low_risk_customer)

In [ ]:
# --- Validation Error Tests ---

print("=== Validation Error Tests ===")

def test_predict_missing_fields():
    """Missing required fields should return 422."""
    resp = test_client.post("/predict", json={
        "tenure_months": 24
        # Missing all other required fields
    })
    assert resp.status_code == 422
    errors = resp.json()["detail"]
    assert len(errors) > 0  # Should have multiple missing field errors

def test_predict_invalid_tenure():
    """Negative tenure should fail validation."""
    resp = test_client.post("/predict", json={
        "tenure_months": -5,
        "monthly_charges": 50,
        "total_charges": 500,
        "contract": "One year",
        "payment_method": "Credit card",
        "has_internet_service": True
    })
    assert resp.status_code == 422

def test_predict_charges_too_high():
    """Monthly charges above limit should fail."""
    resp = test_client.post("/predict", json={
        "tenure_months": 10,
        "monthly_charges": 999.99,  # Exceeds le=500
        "total_charges": 9999,
        "contract": "One year",
        "payment_method": "Credit card",
        "has_internet_service": True
    })
    assert resp.status_code == 422

def test_predict_empty_body():
    """Empty request body should return 422."""
    resp = test_client.post("/predict", json={})
    assert resp.status_code == 422

def test_predict_wrong_types():
    """Wrong types should fail validation."""
    resp = test_client.post("/predict", json={
        "tenure_months": "not_a_number",
        "monthly_charges": 50,
        "total_charges": 500,
        "contract": "One year",
        "payment_method": "Credit card",
        "has_internet_service": True
    })
    assert resp.status_code == 422

run_test("Missing required fields -> 422", test_predict_missing_fields)
run_test("Negative tenure -> 422", test_predict_invalid_tenure)
run_test("Charges too high -> 422", test_predict_charges_too_high)
run_test("Empty body -> 422", test_predict_empty_body)
run_test("Wrong types -> 422", test_predict_wrong_types)

In [ ]:
# --- Edge Case and Batch Tests ---

print("=== Edge Case Tests ===")

def test_predict_boundary_values():
    """Test with minimum valid values."""
    resp = test_client.post("/predict", json={
        "tenure_months": 0,
        "monthly_charges": 0.0,
        "total_charges": 0.0,
        "contract": "Month-to-month",
        "payment_method": "Electronic check",
        "has_internet_service": False,
        "has_tech_support": False,
        "num_dependents": 0
    })
    assert resp.status_code == 200
    assert 0 <= resp.json()["churn_probability"] <= 1

def test_predict_max_values():
    """Test with maximum valid values."""
    resp = test_client.post("/predict", json={
        "tenure_months": 120,
        "monthly_charges": 500.0,
        "total_charges": 60000.0,
        "contract": "Two year",
        "payment_method": "Mailed check",
        "has_internet_service": True,
        "has_tech_support": True,
        "num_dependents": 10
    })
    assert resp.status_code == 200

def test_predict_batch():
    """Batch prediction with multiple customers."""
    customers = []
    for _ in range(5):
        customers.append({
            "tenure_months": int(np.random.randint(0, 72)),
            "monthly_charges": round(float(np.random.uniform(20, 100)), 2),
            "total_charges": round(float(np.random.uniform(100, 5000)), 2),
            "contract": str(np.random.choice(["Month-to-month", "One year", "Two year"])),
            "payment_method": str(np.random.choice(["Electronic check", "Credit card", "Bank transfer", "Mailed check"])),
            "has_internet_service": bool(np.random.choice([True, False])),
            "has_tech_support": bool(np.random.choice([True, False])),
            "num_dependents": int(np.random.randint(0, 5))
        })
    
    resp = test_client.post("/predict/batch", json={"customers": customers})
    assert resp.status_code == 200
    data = resp.json()
    assert data["count"] == 5
    assert len(data["predictions"]) == 5
    assert data["processing_time_ms"] >= 0

def test_predict_batch_empty():
    """Empty batch should return 0 predictions."""
    resp = test_client.post("/predict/batch", json={"customers": []})
    assert resp.status_code == 200
    assert resp.json()["count"] == 0

def test_predict_default_values():
    """Optional fields should use defaults."""
    resp = test_client.post("/predict", json={
        "tenure_months": 12,
        "monthly_charges": 50.0,
        "total_charges": 600.0,
        "contract": "One year",
        "payment_method": "Credit card",
        "has_internet_service": True
        # has_tech_support and num_dependents should use defaults
    })
    assert resp.status_code == 200

run_test("Boundary values (all minimums)", test_predict_boundary_values)
run_test("Boundary values (all maximums)", test_predict_max_values)
run_test("Batch prediction (5 customers)", test_predict_batch)
run_test("Empty batch", test_predict_batch_empty)
run_test("Default optional values", test_predict_default_values)

In [ ]:
# --- Load Testing Concept ---

print("=== Load Testing with concurrent.futures ===")
print("(Simulating concurrent API requests)\n")

def make_single_request(i: int) -> dict:
    """Make a single prediction request and measure latency."""
    start = time.time()
    resp = test_client.post("/predict", json={
        "tenure_months": int(np.random.randint(0, 72)),
        "monthly_charges": round(float(np.random.uniform(20, 100)), 2),
        "total_charges": round(float(np.random.uniform(100, 5000)), 2),
        "contract": str(np.random.choice(["Month-to-month", "One year", "Two year"])),
        "payment_method": str(np.random.choice(["Electronic check", "Credit card"])),
        "has_internet_service": True,
        "has_tech_support": False,
        "num_dependents": 0
    })
    latency_ms = (time.time() - start) * 1000
    return {
        "request_id": i,
        "status_code": resp.status_code,
        "latency_ms": latency_ms
    }

# Run concurrent requests
n_requests = 50
results = []

start_time = time.time()
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(make_single_request, i) for i in range(n_requests)]
    for future in as_completed(futures):
        results.append(future.result())
total_time = time.time() - start_time

# Analyze results
latencies = [r["latency_ms"] for r in results]
statuses = [r["status_code"] for r in results]

print(f"Total requests:    {n_requests}")
print(f"Total time:        {total_time:.2f}s")
print(f"Throughput:        {n_requests / total_time:.1f} req/s")
print(f"Success rate:      {sum(1 for s in statuses if s == 200) / len(statuses):.1%}")
print(f"\nLatency Statistics (ms):")
print(f"  Min:    {min(latencies):.2f}")
print(f"  Mean:   {np.mean(latencies):.2f}")
print(f"  Median: {np.median(latencies):.2f}")
print(f"  P95:    {np.percentile(latencies, 95):.2f}")
print(f"  P99:    {np.percentile(latencies, 99):.2f}")
print(f"  Max:    {max(latencies):.2f}")

# Plot latency distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(latencies, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax1.axvline(np.mean(latencies), color='red', linestyle='--', label=f'Mean: {np.mean(latencies):.1f}ms')
ax1.axvline(np.percentile(latencies, 95), color='orange', linestyle='--', label=f'P95: {np.percentile(latencies, 95):.1f}ms')
ax1.set_xlabel('Latency (ms)')
ax1.set_ylabel('Count')
ax1.set_title('Request Latency Distribution')
ax1.legend()

sorted_latencies = np.sort(latencies)
percentiles = np.arange(1, len(sorted_latencies) + 1) / len(sorted_latencies) * 100
ax2.plot(sorted_latencies, percentiles, color='steelblue', linewidth=2)
ax2.axhline(95, color='orange', linestyle='--', alpha=0.7, label='P95')
ax2.axhline(99, color='red', linestyle='--', alpha=0.7, label='P99')
ax2.set_xlabel('Latency (ms)')
ax2.set_ylabel('Percentile')
ax2.set_title('Latency Percentile Distribution')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Test Results Summary

print("=" * 55)
print("            TEST RESULTS SUMMARY")
print("=" * 55)

passed = sum(1 for _, result in test_results if result == "PASS")
failed = sum(1 for _, result in test_results if result.startswith("FAIL"))
errors = sum(1 for _, result in test_results if result.startswith("ERROR"))

for name, result in test_results:
    icon = "PASS" if result == "PASS" else "FAIL"
    print(f"  [{icon}] {name}")

print(f"\n  Total: {len(test_results)} | Passed: {passed} | Failed: {failed} | Errors: {errors}")
print("=" * 55)

### Exercises

**Exercise 1: Add an `/explain` Endpoint**

Create an endpoint that returns feature importance or SHAP-like explanations for a prediction.

Requirements:
- Accept the same `ChurnFeatures` input as `/predict`
- Return the prediction PLUS a dictionary of feature contributions
- Use the model's `feature_importances_` attribute as a starting point
- Response model should include: prediction, probability, and a `feature_contributions` dict

```python
# Hint: Use model.feature_importances_ weighted by feature values
@app.post("/explain")
def explain(features: ChurnFeatures):
    prediction = make_prediction(features)
    importances = model.feature_importances_
    X = encode_features(features)
    contributions = {
        name: round(float(importances[i] * X[0][i]), 4)
        for i, name in enumerate(feature_columns)
    }
    return {"prediction": prediction, "feature_contributions": contributions}
```

**Exercise 2: Add Rate Limiting Middleware**

Implement simple in-memory rate limiting:
- Track requests per client IP
- Allow maximum 10 requests per minute per IP
- Return 429 (Too Many Requests) when limit exceeded
- Include `Retry-After` header in 429 responses

```python
# Hint: Use a dict to track {ip: [timestamp, timestamp, ...]}
from collections import defaultdict

class RateLimitMiddleware(BaseHTTPMiddleware):
    def __init__(self, app, max_requests: int = 10, window_seconds: int = 60):
        super().__init__(app)
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.requests = defaultdict(list)  # ip -> [timestamps]
    
    async def dispatch(self, request, call_next):
        client_ip = request.client.host
        now = time.time()
        # Clean old entries and check limit
        # ... your implementation here
```

**Exercise 3: Write a Complete Dockerfile**

Write a Dockerfile that:
1. Uses the multi-stage build pattern from Section 2
2. Includes a `requirements.txt` with: `fastapi`, `uvicorn[standard]`, `scikit-learn`, `numpy`, `pandas`
3. Copies the trained model (assume it is saved as `models/churn_model.pkl`)
4. Runs as non-root user
5. Includes a HEALTHCHECK
6. Test by building and running: `docker build -t churn-api . && docker run -p 8000:8000 churn-api`

---

## Section 10: Key Takeaways

### Core Principles

1. **Docker = Reproducibility**: Containers package your entire runtime (OS, Python, libraries, model) into a versioned, portable artifact. The "works on my machine" problem disappears because the machine IS the container.

2. **FastAPI = Modern ML Serving**: FastAPI's type hints, automatic validation, async support, and auto-generated docs make it the ideal framework for ML APIs. Combined with TestClient, you can develop and test without ever starting a server.

3. **Always Validate Inputs with Pydantic**: ML models fail silently on bad inputs. Pydantic catches invalid features at the API boundary with clear error messages, before they reach your model. Use `Field` constraints, custom validators, and enums.

4. **Health Checks Are Essential for Production**: Without `/health`, orchestrators (Kubernetes, ECS) cannot know if your model is loaded and ready. A failed health check triggers automatic restarts or traffic rerouting.

5. **Test Before Deploy**: FastAPI's TestClient makes testing frictionless. Write tests for happy paths, validation errors, edge cases, and batch endpoints. Run them in CI/CD before every deployment.

### Architecture Patterns

| Pattern | When to Use |
|---------|------------|
| Single-stage Dockerfile | Prototyping, simple applications |
| Multi-stage Dockerfile | Production (smaller images, faster deploys) |
| Lifespan model loading | Production (graceful startup/shutdown) |
| Request/response models | Always (type safety + auto docs) |
| Middleware stack | Production (logging, timing, error handling) |
| Docker Compose | Multi-service local development |
| Health + Metrics | Any deployed service |

### What We Built

- A production-ready Dockerfile with multi-stage builds, non-root user, and health checks
- A FastAPI calculator API demonstrating fundamentals
- Pydantic validation models with field constraints, custom validators, and cross-field validation
- A churn prediction API with single and batch endpoints
- A full middleware stack with timing, logging, request IDs, and CORS
- Health check, metrics, and model info endpoints
- A Docker Compose stack with API, Nginx proxy, and Redis cache
- A comprehensive test suite with load testing

### What's Next

In **Notebook 04**, we will cover **MLflow and Experiment Tracking**:
- Tracking experiments (parameters, metrics, artifacts)
- Model registry for versioning and staging
- A/B testing patterns for comparing model versions
- Integrating MLflow with the FastAPI serving pattern from this notebook